
# Saudi Master Data Generator — Self-contained Notebook

This notebook implements the uploaded Saudi master-data generation specification. It creates these DataFrames:

- `territory_df`
- `salesperson_df`
- `van_df`
- `customer_df`
- `holiday_df`
- `config_df`
- `rfm_scores_df`

It also validates foreign keys and business rules, then writes CSV outputs plus `data_quality_report.json`.


In [14]:
# !pip install pandas folium 

In [15]:
# !python -m pip install ortools

In [16]:

# Self-contained Saudi master/reference data generator
# Run this notebook top-to-bottom. It writes outputs into ./saudi_master_data_output

import sys
import subprocess
import importlib.util
from pathlib import Path
from datetime import date, datetime, timedelta
import json
import math
import random
import string


def ensure_package(package_name, import_name=None):
    """Install package if missing. Safe for local/Jupyter execution."""
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("pandas")
ensure_package("numpy")
ensure_package("faker")

import numpy as np
import pandas as pd
from faker import Faker

try:
    fake = Faker(["ar_SA", "en_US"])
except Exception:
    fake = Faker("en_US")

# -----------------------------
# Static specification inputs
# -----------------------------

TERRITORIES = [
    {
        "territory_id": "TER_RUH",
        "territory_name": "Riyadh Central",
        "center_lat": 24.7136,
        "center_lng": 46.6753,
        "radius_km": 25,
        "warehouse_lat": 24.5790,
        "warehouse_lng": 46.8237,
        "warehouse_address": "Industrial Area, Riyadh",
    },
    {
        "territory_id": "TER_JED",
        "territory_name": "Jeddah North",
        "center_lat": 21.5433,
        "center_lng": 39.1728,
        "radius_km": 22,
        "warehouse_lat": 21.3429,
        "warehouse_lng": 39.2357,
        "warehouse_address": "Al Khomrah Logistics Area, Jeddah",
    },
    {
        "territory_id": "TER_DMM",
        "territory_name": "Dammam Metro",
        "center_lat": 26.4207,
        "center_lng": 50.0888,
        "radius_km": 20,
        "warehouse_lat": 26.2926,
        "warehouse_lng": 50.1629,
        "warehouse_address": "2nd Industrial City, Dammam",
    },
]

LOCALITIES = {
    "TER_RUH": [
        ("Olaya", 24.7115, 46.6746),
        ("Al Malaz", 24.6676, 46.7351),
        ("Al Sulaymaniyah", 24.7012, 46.7112),
        ("Al Yasmin", 24.8271, 46.6302),
        ("Hittin", 24.7636, 46.6022),
    ],
    "TER_JED": [
        ("Al Rawdah", 21.5656, 39.1652),
        ("Al Hamra", 21.5262, 39.1611),
        ("Al Safa", 21.5854, 39.2181),
        ("Al Salamah", 21.5948, 39.1485),
        ("Al Zahra", 21.6152, 39.1335),
    ],
    "TER_DMM": [
        ("Al Faisaliyah", 26.4282, 50.0786),
        ("Al Shati", 26.4701, 50.1124),
        ("Al Mazruiyah", 26.4481, 50.0962),
        ("Al Badiyah", 26.4021, 50.0587),
        ("Al Nuzha", 26.4337, 50.0433),
    ],
}

CITY_CODES = {"TER_RUH": "RUH", "TER_JED": "JED", "TER_DMM": "DMM"}
PLATE_PREFIXES = {"TER_RUH": "RU", "TER_JED": "JE", "TER_DMM": "DM"}

SAUDI_SALESPERSON_NAMES = [
    "Abdullah Al-Qahtani", "Fahad Al-Otaibi", "Mohammed Al-Harbi",
    "Nasser Al-Dossari", "Khalid Al-Ghamdi", "Saeed Al-Zahrani",
    "Yousef Al-Mutairi", "Majed Al-Shammari", "Ahmed Al-Anazi",
    "Salem Al-Rashidi", "Omar Al-Shehri", "Hassan Al-Yami",
    "Rashid Al-Malki", "Ibrahim Al-Subaie", "Mansour Al-Qahtani",
    "Waleed Al-Harbi", "Bilal Khan", "Imran Ahmed", "Sameer Khan",
    "Nadeem Ali", "Arif Rahman", "Mustafa Hussain",
]

OWNER_NAMES = [
    "Al Rajhi", "Al Othman", "Al Harbi", "Al Qahtani", "Al Ghamdi",
    "Al Zahrani", "Al Dossari", "Al Mutairi", "Al Shammari", "Al Anazi",
    "Al Rashid", "Al Saleh", "Al Malki", "Al Subaie", "Al Shehri",
]

BUSINESS_PREFIXES = [
    "Al Noor", "Al Waha", "Al Baraka", "Al Safa", "Al Madina",
    "Al Qassim", "Al Riyadh", "Al Jazeera", "Al Khaleej", "Al Nada",
    "Al Nakheel", "Al Rawabi", "Al Tazaj", "Al Dana", "Al Manar",
]

SHOP_CATEGORIES = [
    "Grocery", "Mini Market", "Supermarket", "Restaurant", "Cafe", "Bakery",
    "Butchery", "Cold Store", "Hotel Kitchen", "Catering Kitchen",
]

BUSINESS_SUFFIX_BY_CATEGORY = {
    "Grocery": ["Grocery", "Baqala", "Food Store"],
    "Mini Market": ["Mini Market", "Corner Market", "Baqala"],
    "Supermarket": ["Supermarket", "Hyper Mini", "Market"],
    "Restaurant": ["Restaurant", "Kitchen", "Grill"],
    "Cafe": ["Cafe", "Coffee House", "Roastery"],
    "Bakery": ["Bakery", "Sweets & Bakery", "Oven"],
    "Butchery": ["Butchery", "Meat Shop", "Fresh Meat"],
    "Cold Store": ["Cold Store", "Frozen Foods", "Chilled Foods"],
    "Hotel Kitchen": ["Hotel Kitchen", "Hospitality Supplies"],
    "Catering Kitchen": ["Catering Kitchen", "Banquet Kitchen"],
}

VISIT_DAYS = ["Saturday", "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday"]
ORDER_WINDOWS = ["Morning", "Midday", "Afternoon"]
LIFECYCLE_STATES = ["Active", "New", "At Risk", "Dormant", "Churned"]
LIFECYCLE_PROBS = [0.65, 0.10, 0.15, 0.08, 0.02]

# -----------------------------
# Utility functions
# -----------------------------

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    Faker.seed(seed)


def haversine_km(lat1, lng1, lat2, lng2):
    """Great-circle distance without external geospatial dependencies."""
    r = 6371.0088
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lng2 - lng1)
    a = math.sin(dphi/2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda/2)**2
    return 2 * r * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def jitter_location(lat, lng, radius_km=2.5):
    lat_jitter = np.random.normal(0, radius_km / 111 / 2)
    lng_jitter = np.random.normal(0, radius_km / (111 * np.cos(np.radians(lat))) / 2)
    return float(lat + lat_jitter), float(lng + lng_jitter)


def random_date_between(start, end):
    days = (end - start).days
    return start + timedelta(days=random.randint(0, days))


def weighted_choice(values, probs):
    return random.choices(values, weights=probs, k=1)[0]


def generate_plate(prefix):
    return f"{prefix}{random.choice(string.ascii_uppercase)} {random.randint(1000, 9999)}"


def performance_multiplier():
    bucket = random.random()
    if bucket < 0.15:
        return round(random.uniform(1.10, 1.20), 2)
    if bucket < 0.85:
        return round(random.uniform(0.95, 1.08), 2)
    return round(random.uniform(0.85, 0.94), 2)


def cold_truck_required(category):
    probs = {
        "Cold Store": 1.00,
        "Butchery": 1.00,
        "Hotel Kitchen": 0.80,
        "Catering Kitchen": 0.75,
        "Restaurant": 0.60,
        "Supermarket": 0.45,
        "Bakery": 0.25,
        "Cafe": 0.20,
        "Grocery": 0.12,
        "Mini Market": 0.12,
    }
    return random.random() < probs[category]


def choose_payment_type(volume_tier):
    credit_prob = {"HIGH": 0.75, "MED": 0.45, "LOW": 0.20}[volume_tier]
    return "credit" if random.random() < credit_prob else "cash"


def generate_credit(volume_tier, payment_type, lifecycle_state):
    if payment_type == "cash":
        return 0.0, 0.0
    limit_ranges = {
        "HIGH": (30000, 120000),
        "MED": (10000, 45000),
        "LOW": (2000, 15000),
    }
    credit_limit = round(random.uniform(*limit_ranges[volume_tier]), 2)
    if lifecycle_state in ["At Risk", "Dormant"]:
        pct = random.uniform(0.55, 1.10)
    elif lifecycle_state == "Churned":
        pct = random.uniform(0.70, 1.10)
    else:
        pct = random.choices(
            [random.uniform(0.05, 0.35), random.uniform(0.35, 0.70), random.uniform(0.70, 1.10)],
            weights=[0.65, 0.25, 0.10],
            k=1,
        )[0]
    return credit_limit, round(credit_limit * pct, 2)


def generate_shop_name(category, locality, used_names):
    suffix = random.choice(BUSINESS_SUFFIX_BY_CATEGORY[category])
    include_locality = random.random() < 0.28
    templates = [
        "{prefix} {suffix}",
        "{owner} {suffix}",
        "{prefix} Fresh {suffix}",
        "{owner} Trading {suffix}",
    ]
    if include_locality:
        templates.extend(["{locality} {suffix}", "{prefix} {locality} {suffix}"])

    # Keep cold-chain suffixes aligned with cold-chain categories.
    if suffix in {"Frozen Foods", "Cold Store", "Chilled Foods"} and category != "Cold Store":
        suffix = random.choice(BUSINESS_SUFFIX_BY_CATEGORY[category])

    for _ in range(50):
        name = random.choice(templates).format(
            prefix=random.choice(BUSINESS_PREFIXES),
            owner=random.choice(OWNER_NAMES),
            locality=locality,
            suffix=suffix,
        )
        if name not in used_names:
            used_names.add(name)
            return name
    name = f"{locality} {suffix} {random.randint(100, 999)}"
    used_names.add(name)
    return name

# -----------------------------
# Generator functions
# -----------------------------

def generate_territories():
    df = pd.DataFrame(TERRITORIES).copy()
    df["default_salesperson"] = None
    df["default_van"] = None
    return df[
        ["territory_id", "territory_name", "warehouse_lat", "warehouse_lng", "warehouse_address",
         "center_lat", "center_lng", "radius_km", "default_salesperson", "default_van"]
    ]


def generate_salespeople(territory_df, salespeople_per_territory=3):
    names = SAUDI_SALESPERSON_NAMES.copy()
    random.shuffle(names)
    rows = []
    for _, ter in territory_df.iterrows():
        code = CITY_CODES[ter.territory_id]
        for i in range(1, salespeople_per_territory + 1):
            rows.append({
                "sales_id": f"SAL_{code}_{i:03d}",
                "name": names.pop() if names else fake.name(),
                "territory_id": ter.territory_id,
                "working_hours_per_day": 8.0,
                "shift_start_time": "09:00",
                "assigned_van": None,
                "active_status": random.random() < 0.93,
                "performance_multiplier": performance_multiplier(),
            })
    return pd.DataFrame(rows)


def generate_vans(territory_df, salesperson_df, backup_vans_per_territory=1):
    rows = []
    for _, ter in territory_df.iterrows():
        n_sales = int((salesperson_df["territory_id"] == ter.territory_id).sum())
        total_vans = n_sales + backup_vans_per_territory
        code = CITY_CODES[ter.territory_id]
        prefix = PLATE_PREFIXES[ter.territory_id]
        for i in range(1, total_vans + 1):
            rows.append({
                "van_id": f"VAN_{code}_{i:03d}",
                "registration_no": generate_plate(prefix),
                "cold_truck_enabled": random.random() < 0.50,
                "territory_id": ter.territory_id,
                "active_status": random.random() < 0.96,
            })
    return pd.DataFrame(rows)


def assign_vans_to_salespeople(salesperson_df, van_df):
    salesperson_df = salesperson_df.copy()
    for territory_id in salesperson_df["territory_id"].unique():
        sp_idx = salesperson_df.index[salesperson_df["territory_id"] == territory_id].tolist()
        vans = van_df[van_df["territory_id"] == territory_id]["van_id"].tolist()
        for idx, van_id in zip(sp_idx, vans):
            salesperson_df.loc[idx, "assigned_van"] = van_id
    return salesperson_df


def assign_defaults(territory_df, salesperson_df, van_df):
    territory_df = territory_df.copy()
    for idx, ter in territory_df.iterrows():
        territory_id = ter.territory_id
        active_sp = salesperson_df[(salesperson_df.territory_id == territory_id) & (salesperson_df.active_status)]
        sp_pool = active_sp if not active_sp.empty else salesperson_df[salesperson_df.territory_id == territory_id]
        default_sp = sp_pool.sort_values("performance_multiplier", ascending=False).iloc[0]
        default_van = default_sp.assigned_van
        if pd.isna(default_van):
            default_van = van_df[van_df.territory_id == territory_id].iloc[0].van_id
        territory_df.loc[idx, "default_salesperson"] = default_sp.sales_id
        territory_df.loc[idx, "default_van"] = default_van
    return territory_df


def generate_customers(territory_df, customers_per_territory=100):
    rows = []
    today = date.today()
    start_acq = today - timedelta(days=5 * 365)
    end_acq = today - timedelta(days=30)

    for _, ter in territory_df.iterrows():
        territory_id = ter.territory_id
        code = CITY_CODES[territory_id]
        used_names = set()
        tiers = ["HIGH"] * 20 + ["MED"] * 30 + ["LOW"] * 50
        random.shuffle(tiers)

        for i in range(1, customers_per_territory + 1):
            locality, base_lat, base_lng = random.choice(LOCALITIES[territory_id])
            for _ in range(100):
                lat, lng = jitter_location(base_lat, base_lng, radius_km=2.5)
                if haversine_km(lat, lng, ter.center_lat, ter.center_lng) <= ter.radius_km:
                    break
            else:
                lat, lng = base_lat, base_lng

            tier = tiers[i - 1]
            lifecycle = weighted_choice(LIFECYCLE_STATES, LIFECYCLE_PROBS)
            category = random.choice(SHOP_CATEGORIES)
            payment_type = choose_payment_type(tier)
            credit_limit, outstanding_balance = generate_credit(tier, payment_type, lifecycle)
            customer_rating = int(random.choices([1, 2, 3, 4, 5], weights=[0.05, 0.10, 0.25, 0.35, 0.25], k=1)[0])

            rows.append({
                "customer_id": f"CUS_{code}_{i:04d}",
                "shop_name": generate_shop_name(category, locality, used_names),
                "gps_lat": round(lat, 6),
                "gps_lng": round(lng, 6),
                "locality": locality,
                "territory_id": territory_id,
                "customer_rating": customer_rating,
                "review_rating": round(float(np.clip(np.random.normal(4.0, 0.55), 2.5, 5.0)), 1),
                "shop_category": category,
                "cold_truck_required": cold_truck_required(category),
                "volume_tier": tier,
                "payment_type": payment_type,
                "credit_limit": credit_limit,
                "outstanding_balance": outstanding_balance,
                "lifecycle_state": lifecycle,
                "acquisition_date": random_date_between(start_acq, end_acq),
                "preferred_visit_day": random.choice(VISIT_DAYS),
                "preferred_order_window": random.choice(ORDER_WINDOWS),
            })
    return pd.DataFrame(rows)


def fridays_between(start_date, end_date):
    cur = start_date
    while cur.weekday() != 4:  # Monday=0, Friday=4
        cur += timedelta(days=1)
    while cur <= end_date:
        yield cur
        cur += timedelta(days=7)


def generate_holidays(territory_df, salesperson_df, van_df, months=3):
    rows = []
    today = date.today()
    start = date(today.year, today.month, 1)
    end = start + timedelta(days=months * 31)
    holiday_id = 1

    # Territory Friday closure rows
    for _, ter in territory_df.iterrows():
        for d in fridays_between(start, end):
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": None,
                "van_holiday": None,
                "territory_holiday": ter.territory_id,
                "from_date": d,
                "to_date": d,
                "reason": "Friday closure",
            })
            holiday_id += 1

        # Common Saudi master calendar events. Dates are approximate placeholders for synthetic data.
        for label, offset_days, duration_days in [
            ("Ramadan adjusted operating hours", 10, 29),
            ("Eid Al-Fitr holiday", 40, 4),
            ("Saudi National Day", 145, 1),
        ]:
            from_d = start + timedelta(days=offset_days)
            to_d = from_d + timedelta(days=duration_days - 1)
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": None,
                "van_holiday": None,
                "territory_holiday": ter.territory_id,
                "from_date": from_d,
                "to_date": to_d,
                "reason": label,
            })
            holiday_id += 1

    # Salesperson leave: 1-2 short leaves each
    for _, sp in salesperson_df.iterrows():
        for _ in range(random.randint(1, 2)):
            from_d = random_date_between(start, end)
            duration = random.choice([1, 1, 2, 3])
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": sp.sales_id,
                "van_holiday": None,
                "territory_holiday": None,
                "from_date": from_d,
                "to_date": from_d + timedelta(days=duration - 1),
                "reason": random.choice(["Annual leave", "Sick leave"]),
            })
            holiday_id += 1

    # Van maintenance: 1 maintenance day per van per month
    for _, van in van_df.iterrows():
        for month_offset in range(months):
            month_start = start + timedelta(days=month_offset * 31)
            from_d = month_start + timedelta(days=random.randint(0, 27))
            rows.append({
                "holiday_id": f"HOL_{holiday_id:05d}",
                "salesperson_holiday": None,
                "van_holiday": van.van_id,
                "territory_holiday": None,
                "from_date": from_d,
                "to_date": from_d,
                "reason": "Van maintenance",
            })
            holiday_id += 1

    return pd.DataFrame(rows)


def generate_config():
    values = {
        "avg_speed_kmh": 32,
        "avg_service_time_min": 22,
        "buffer_pct": 0.15,
        "rfm_window_days": 90,
        "route_partial_prob": 0.08,
        "route_cancel_prob": 0.03,
        "traffic_jam_prob": 0.12,
        "credit_outstanding_cap": 0.85,
        "normal_shift_start_time": "09:00",
        "ramadan_shift_start_time": "10:00",
    }
    return pd.DataFrame([{"config_key": k, "config_value": v} for k, v in values.items()])


def score_from_quantiles(series, higher_is_better=True):
    # qcut can fail on repeated values, so rank first for stable 1-5 scores.
    ranks = series.rank(method="first")
    labels = [1, 2, 3, 4, 5]
    scored = pd.qcut(ranks, q=5, labels=labels).astype(int)
    if not higher_is_better:
        scored = 6 - scored
    return scored


def generate_rfm_scores(customer_df):
    rows = []
    for _, c in customer_df.iterrows():
        tier = c.volume_tier
        lifecycle = c.lifecycle_state

        if tier == "HIGH":
            recency = random.randint(1, 20)
            frequency = random.randint(20, 50)
            monetary = random.uniform(25000, 180000)
        elif tier == "MED":
            recency = random.randint(7, 45)
            frequency = random.randint(8, 25)
            monetary = random.uniform(7000, 55000)
        else:
            recency = random.randint(20, 90)
            frequency = random.randint(1, 12)
            monetary = random.uniform(500, 12000)

        if lifecycle == "New":
            recency = random.randint(1, 14)
            frequency = max(1, int(frequency * random.uniform(0.25, 0.55)))
            monetary *= random.uniform(0.25, 0.60)
        elif lifecycle == "At Risk":
            recency = max(recency, random.randint(45, 100))
            frequency = max(1, int(frequency * random.uniform(0.45, 0.85)))
            monetary *= random.uniform(0.60, 1.00)
        elif lifecycle == "Dormant":
            recency = random.randint(90, 180)
            frequency = max(0, int(frequency * random.uniform(0.10, 0.35)))
            monetary *= random.uniform(0.10, 0.35)
        elif lifecycle == "Churned":
            recency = random.randint(181, 365)
            frequency = random.choice([0, 0, 1])
            monetary *= random.uniform(0.00, 0.10)

        rows.append({
            "customer_id": c.customer_id,
            "recency": int(recency),
            "frequency": int(frequency),
            "monetary": round(float(monetary), 2),
        })

    df = pd.DataFrame(rows)
    df["r_score"] = score_from_quantiles(df["recency"], higher_is_better=False)
    df["f_score"] = score_from_quantiles(df["frequency"], higher_is_better=True)
    df["m_score"] = score_from_quantiles(df["monetary"], higher_is_better=True)
    df["rfm_score"] = df["r_score"].astype(str) + df["f_score"].astype(str) + df["m_score"].astype(str)

    def segment(row):
        r, f, m = row.r_score, row.f_score, row.m_score
        if r >= 4 and f >= 4 and m >= 4:
            return "Champion"
        if f >= 4 and m >= 3:
            return "Loyal"
        if r >= 4 and f in [2, 3]:
            return "Potential Loyalist"
        if r <= 2 and f >= 3:
            return "At Risk"
        if r <= 2 and f <= 2 and m <= 2:
            return "Hibernating"
        return "Need Attention"

    df["segment"] = df.apply(segment, axis=1)
    return df

# -----------------------------
# Validation and reporting
# -----------------------------

def assert_fk(child_df, child_col, parent_df, parent_col):
    missing = set(child_df[child_col].dropna()) - set(parent_df[parent_col])
    assert not missing, f"Broken FK {child_col}: {missing}"


def validate_all(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df):
    # Foreign keys
    assert_fk(customer_df, "territory_id", territory_df, "territory_id")
    assert_fk(salesperson_df, "territory_id", territory_df, "territory_id")
    assert_fk(van_df, "territory_id", territory_df, "territory_id")
    assert_fk(holiday_df, "territory_holiday", territory_df, "territory_id")
    assert_fk(holiday_df, "salesperson_holiday", salesperson_df, "sales_id")
    assert_fk(holiday_df, "van_holiday", van_df, "van_id")
    assert_fk(rfm_scores_df, "customer_id", customer_df, "customer_id")

    # Primary keys
    assert territory_df["territory_id"].is_unique
    assert salesperson_df["sales_id"].is_unique
    assert van_df["van_id"].is_unique
    assert customer_df["customer_id"].is_unique
    assert holiday_df["holiday_id"].is_unique
    assert rfm_scores_df["customer_id"].is_unique

    # Row counts
    assert len(territory_df) == 3
    assert len(customer_df) == 300
    assert 6 <= len(salesperson_df) <= 9
    assert 9 <= len(van_df) <= 12
    assert len(rfm_scores_df) == 300

    # Same territory checks
    van_territory = van_df.set_index("van_id")["territory_id"].to_dict()
    sp_territory = salesperson_df.set_index("sales_id")["territory_id"].to_dict()
    for _, sp in salesperson_df.iterrows():
        assert van_territory[sp.assigned_van] == sp.territory_id
    for _, ter in territory_df.iterrows():
        assert sp_territory[ter.default_salesperson] == ter.territory_id
        assert van_territory[ter.default_van] == ter.territory_id

    # Business rules
    cash = customer_df["payment_type"] == "cash"
    assert (customer_df.loc[cash, "credit_limit"] == 0).all()
    assert (customer_df.loc[cash, "outstanding_balance"] == 0).all()
    credit = customer_df["payment_type"] == "credit"
    assert (customer_df.loc[credit, "credit_limit"] > 0).all()
    assert customer_df.loc[customer_df.shop_category.isin(["Cold Store", "Butchery"]), "cold_truck_required"].all()

    # Customer coordinates inside territory radius
    ter_lookup = territory_df.set_index("territory_id")
    for _, c in customer_df.iterrows():
        ter = ter_lookup.loc[c.territory_id]
        dist = haversine_km(c.gps_lat, c.gps_lng, ter.center_lat, ter.center_lng)
        assert dist <= ter.radius_km + 1e-6, f"Customer outside radius: {c.customer_id} {dist:.2f}km"

    # No duplicate shop names inside territory
    dupes = customer_df.duplicated(subset=["territory_id", "shop_name"]).sum()
    assert dupes == 0, f"Duplicate shop names inside territory: {dupes}"

    # Tier distribution
    tier_counts = customer_df.groupby(["territory_id", "volume_tier"]).size().unstack(fill_value=0)
    for territory_id in territory_df["territory_id"]:
        assert int(tier_counts.loc[territory_id, "HIGH"]) == 20
        assert int(tier_counts.loc[territory_id, "MED"]) == 30
        assert int(tier_counts.loc[territory_id, "LOW"]) == 50

    return True


def data_quality_report(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df):
    report = {
        "row_counts": {
            "territory": len(territory_df),
            "salesperson": len(salesperson_df),
            "van": len(van_df),
            "customer": len(customer_df),
            "holiday": len(holiday_df),
            "config": len(config_df),
            "rfm_scores": len(rfm_scores_df),
        },
        "customer_tiers_by_territory": customer_df.groupby(["territory_id", "volume_tier"]).size().unstack(fill_value=0).to_dict(orient="index"),
        "lifecycle_distribution": customer_df["lifecycle_state"].value_counts(normalize=True).round(3).to_dict(),
        "payment_type_distribution": customer_df["payment_type"].value_counts(normalize=True).round(3).to_dict(),
        "cold_truck_required_share": round(float(customer_df["cold_truck_required"].mean()), 3),
        "rfm_segments": rfm_scores_df["segment"].value_counts().to_dict(),
        "validation_status": "passed",
        "generated_at": datetime.now().isoformat(timespec="seconds"),
    }
    return report


def generate_all(seed=42, output_dir="saudi_master_data_output", write_files=True):
    set_seed(seed)

    territory_df = generate_territories()
    salesperson_df = generate_salespeople(territory_df)
    van_df = generate_vans(territory_df, salesperson_df)
    salesperson_df = assign_vans_to_salespeople(salesperson_df, van_df)
    territory_df = assign_defaults(territory_df, salesperson_df, van_df)
    customer_df = generate_customers(territory_df)
    holiday_df = generate_holidays(territory_df, salesperson_df, van_df)
    config_df = generate_config()
    rfm_scores_df = generate_rfm_scores(customer_df)

    validate_all(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df)
    report = data_quality_report(territory_df, salesperson_df, van_df, customer_df, holiday_df, config_df, rfm_scores_df)

    outputs = {
        "territory": territory_df,
        "salesperson": salesperson_df,
        "van": van_df,
        "customer": customer_df,
        "holiday": holiday_df,
        "config": config_df,
        "rfm_scores": rfm_scores_df,
    }

    if write_files:
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        for name, df in outputs.items():
            df.to_csv(out / f"{name}.csv", index=False)
        with open(out / "data_quality_report.json", "w", encoding="utf-8") as f:
            json.dump(report, f, indent=2, default=str, ensure_ascii=False)
        print(f"Wrote CSV/JSON outputs to: {out.resolve()}")

    return outputs, report

# Generate data
outputs, report = generate_all(seed=42)
territory_df = outputs["territory"]
salesperson_df = outputs["salesperson"]
van_df = outputs["van"]
customer_df = outputs["customer"]
holiday_df = outputs["holiday"]
config_df = outputs["config"]
rfm_scores_df = outputs["rfm_scores"]

print(json.dumps(report, indent=2, default=str, ensure_ascii=False))


Wrote CSV/JSON outputs to: D:\Data Science\Basamh\JP\journey-planner\saudi_master_data_output
{
  "row_counts": {
    "territory": 3,
    "salesperson": 9,
    "van": 12,
    "customer": 300,
    "holiday": 98,
    "config": 10,
    "rfm_scores": 300
  },
  "customer_tiers_by_territory": {
    "TER_DMM": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    },
    "TER_JED": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    },
    "TER_RUH": {
      "HIGH": 20,
      "LOW": 50,
      "MED": 30
    }
  },
  "lifecycle_distribution": {
    "Active": 0.65,
    "New": 0.123,
    "At Risk": 0.12,
    "Dormant": 0.103,
    "Churned": 0.003
  },
  "payment_type_distribution": {
    "cash": 0.637,
    "credit": 0.363
  },
  "cold_truck_required_share": 0.523,
  "rfm_segments": {
    "Need Attention": 76,
    "Champion": 69,
    "Hibernating": 47,
    "Loyal": 41,
    "At Risk": 37,
    "Potential Loyalist": 30
  },
  "validation_status": "passed",
  "generated_at": "2026-05-27T15:15:


## Preview generated tables

Run the cells below after generation to inspect samples and distribution checks.


In [17]:
holiday_df[holiday_df["salesperson_holiday"].notna()]

,holiday_id,salesperson_holiday,van_holiday,territory_holiday,from_date,to_date,reason
51,HOL_00052,SAL_RUH_001,NaN,NaN,2026-07-03,2026-07-03,Annual leave
52,HOL_00053,SAL_RUH_002,NaN,NaN,2026-06-03,2026-06-03,Annual leave
53,HOL_00054,SAL_RUH_003,NaN,NaN,2026-06-01,2026-06-02,Annual leave
54,HOL_00055,SAL_JED_001,NaN,NaN,2026-05-16,2026-05-16,Sick leave
55,HOL_00056,SAL_JED_002,NaN,NaN,2026-07-09,2026-07-09,Sick leave
56,HOL_00057,SAL_JED_003,NaN,NaN,2026-07-19,2026-07-20,Sick leave
57,HOL_00058,SAL_JED_003,NaN,NaN,2026-07-13,2026-07-14,Sick leave
58,HOL_00059,SAL_DMM_001,NaN,NaN,2026-07-05,2026-07-06,Sick leave
59,HOL_00060,SAL_DMM_002,NaN,NaN,2026-07-13,2026-07-13,Annual leave
60,HOL_00061,SAL_DMM_003,NaN,NaN,2026-07-01,2026-07-01,Sick leave


In [18]:

print("Territories")
display(territory_df)

print("Salespeople")
display(salesperson_df.head(10))

print("Vans")
display(van_df.head(10))

print("Customers")
display(customer_df.head(10))

print("RFM scores")
display(rfm_scores_df.head(10))


Territories


,territory_id,territory_name,warehouse_lat,warehouse_lng,warehouse_address,center_lat,center_lng,radius_km,default_salesperson,default_van
0,TER_RUH,Riyadh Central,24.5790,46.8237,"Industrial Area, Riyadh",24.7136,46.6753,25,SAL_RUH_001,VAN_RUH_001
1,TER_JED,Jeddah North,21.3429,39.2357,"Al Khomrah Logistics Area, Jeddah",21.5433,39.1728,22,SAL_JED_002,VAN_JED_002
2,TER_DMM,Dammam Metro,26.2926,50.1629,"2nd Industrial City, Dammam",26.4207,50.0888,20,SAL_DMM_002,VAN_DMM_002


Salespeople


,sales_id,name,territory_id,working_hours_per_day,shift_start_time,assigned_van,active_status,performance_multiplier
0,SAL_RUH_001,Arif Rahman,TER_RUH,8.0,09:00,VAN_RUH_001,True,1.04
1,SAL_RUH_002,Nasser Al-Dossari,TER_RUH,8.0,09:00,VAN_RUH_002,True,0.99
2,SAL_RUH_003,Abdullah Al-Qahtani,TER_RUH,8.0,09:00,VAN_RUH_003,True,0.97
3,SAL_JED_001,Ahmed Al-Anazi,TER_JED,8.0,09:00,VAN_JED_001,True,0.98
4,SAL_JED_002,Majed Al-Shammari,TER_JED,8.0,09:00,VAN_JED_002,True,1.14
5,SAL_JED_003,Imran Ahmed,TER_JED,8.0,09:00,VAN_JED_003,True,0.98
6,SAL_DMM_001,Khalid Al-Ghamdi,TER_DMM,8.0,09:00,VAN_DMM_001,True,0.97
7,SAL_DMM_002,Hassan Al-Yami,TER_DMM,8.0,09:00,VAN_DMM_002,True,1.13
8,SAL_DMM_003,Fahad Al-Otaibi,TER_DMM,8.0,09:00,VAN_DMM_003,True,0.88


Vans


,van_id,registration_no,cold_truck_enabled,territory_id,active_status
0,VAN_RUH_001,RUG 2139,True,TER_RUH,True
1,VAN_RUH_002,RUJ 2307,False,TER_RUH,True
2,VAN_RUH_003,RUM 5554,True,TER_RUH,True
3,VAN_RUH_004,RUF 7065,True,TER_RUH,True
4,VAN_JED_001,JEW 2169,False,TER_JED,True
5,VAN_JED_002,JEX 5010,True,TER_JED,True
6,VAN_JED_003,JEU 4598,False,TER_JED,True
7,VAN_JED_004,JEY 1916,True,TER_JED,True
8,VAN_DMM_001,DMK 7572,True,TER_DMM,True
9,VAN_DMM_002,DMS 6155,True,TER_DMM,True


Customers


,customer_id,shop_name,gps_lat,gps_lng,locality,territory_id,customer_rating,review_rating,shop_category,cold_truck_required,volume_tier,payment_type,credit_limit,outstanding_balance,lifecycle_state,acquisition_date,preferred_visit_day,preferred_order_window
0,CUS_RUH_0001,Al Waha Fresh Mini Market,24.769194,46.600485,Hittin,TER_RUH,2,4.4,Mini Market,False,LOW,cash,0.00,0.00,Active,2025-02-26,Tuesday,Morning
1,CUS_RUH_0002,Al Othman Banquet Kitchen,24.780751,46.599296,Hittin,TER_RUH,3,3.9,Catering Kitchen,True,MED,cash,0.00,0.00,Active,2023-05-23,Tuesday,Midday
2,CUS_RUH_0003,Al Qassim Al Yasmin Grocery,24.844884,46.639722,Al Yasmin,TER_RUH,5,3.7,Grocery,False,MED,cash,0.00,0.00,At Risk,2022-10-19,Sunday,Morning
3,CUS_RUH_0004,Al Waha Fresh Meat,24.769710,46.596453,Hittin,TER_RUH,2,3.7,Butchery,True,HIGH,credit,71636.53,26962.89,Active,2022-09-24,Sunday,Midday
4,CUS_RUH_0005,Al Manar Fresh Butchery,24.829825,46.606460,Al Yasmin,TER_RUH,3,3.1,Butchery,True,MED,cash,0.00,0.00,Active,2023-01-02,Tuesday,Afternoon
5,CUS_RUH_0006,Al Shammari Chilled Foods,24.757268,46.589639,Hittin,TER_RUH,2,4.2,Cold Store,True,LOW,credit,5857.20,3237.35,New,2022-06-12,Saturday,Afternoon
6,CUS_RUH_0007,Olaya Grill,24.701275,46.657092,Olaya,TER_RUH,5,4.8,Restaurant,True,LOW,cash,0.00,0.00,New,2025-02-02,Wednesday,Afternoon
7,CUS_RUH_0008,Hittin Cafe,24.761057,46.603037,Hittin,TER_RUH,3,3.2,Cafe,True,MED,credit,35066.71,4264.31,Active,2022-08-07,Wednesday,Midday
8,CUS_RUH_0009,Al Nada Fresh Mini Market,24.661470,46.736475,Al Malaz,TER_RUH,3,3.4,Mini Market,False,LOW,cash,0.00,0.00,Dormant,2021-06-13,Thursday,Afternoon
9,CUS_RUH_0010,Al Ghamdi Mini Market,24.705431,46.703755,Al Sulaymaniyah,TER_RUH,2,3.8,Mini Market,False,MED,cash,0.00,0.00,Dormant,2025-06-05,Monday,Morning


RFM scores


,customer_id,recency,frequency,monetary,r_score,f_score,m_score,rfm_score,segment
0,CUS_RUH_0001,71,5,5683.38,2,2,2,222,Hibernating
1,CUS_RUH_0002,43,12,54005.08,3,4,5,345,Loyal
2,CUS_RUH_0003,96,5,32151.46,1,2,4,124,Need Attention
3,CUS_RUH_0004,13,26,47605.78,4,5,5,455,Champion
4,CUS_RUH_0005,16,23,45672.08,4,5,5,455,Champion
5,CUS_RUH_0006,14,3,2728.57,4,1,1,411,Need Attention
6,CUS_RUH_0007,9,4,1053.56,5,2,1,521,Potential Loyalist
7,CUS_RUH_0008,13,13,9881.75,4,4,3,443,Loyal
8,CUS_RUH_0009,102,1,1863.86,1,1,1,111,Hibernating
9,CUS_RUH_0010,122,3,6879.02,1,1,2,112,Hibernating


In [19]:

print("Tier counts per territory")
display(customer_df.groupby(["territory_id", "volume_tier"]).size().unstack(fill_value=0))

print("Lifecycle distribution")
display(customer_df["lifecycle_state"].value_counts(normalize=True).rename("share").round(3))

print("RFM segments")
display(rfm_scores_df["segment"].value_counts())

print("Output files")
for path in sorted(Path("saudi_master_data_output").glob("*")):
    print(path)


Tier counts per territory


volume_tier,HIGH,LOW,MED
territory_id,,,
TER_DMM,20,50,30
TER_JED,20,50,30
TER_RUH,20,50,30


Lifecycle distribution


lifecycle_state
Active     0.650
New        0.123
At Risk    0.120
Dormant    0.103
Churned    0.003
Name: share, dtype: float64

RFM segments


segment
Need Attention        76
Champion              69
Hibernating           47
Loyal                 41
At Risk               37
Potential Loyalist    30
Name: count, dtype: int64

Output files
saudi_master_data_output\config.csv
saudi_master_data_output\customer.csv
saudi_master_data_output\data_quality_report.json
saudi_master_data_output\holiday.csv
saudi_master_data_output\rfm_scores.csv
saudi_master_data_output\salesperson.csv
saudi_master_data_output\territory.csv
saudi_master_data_output\van.csv



## Notes

- This notebook intentionally excludes route, visit, and basket-level transaction tables.
- The holiday dates are synthetic placeholders suitable for master-data demos. For production Saudi holiday calendars, replace those placeholder rows with an official calendar source.
- `generate_all(seed=42)` is deterministic for repeatable demos. Change the seed to create a different synthetic dataset.


In [24]:
from saudi_multi_salesperson_scheduler import MultiSalespersonScheduler

result = MultiSalespersonScheduler().create_schedules(
    customer_df=customer_df,
    rfm_scores_df=rfm_scores_df,
    salesperson_df=salesperson_df,
    holiday_df=holiday_df,
    territory_df=territory_df,
    config_df=config_df,
    van_df=van_df,
    month_start_date="2026-07-01",
)

Status: OPTIMAL
Objective value: 101879093.0
Best bound: 101879093.0
Status: OPTIMAL
Objective value: 43551226.0
Best bound: 43551226.0
Status: OPTIMAL
Objective value: 94784177.0
Best bound: 94784177.0
Status: OPTIMAL
Objective value: 50452288.0
Best bound: 50452288.0
Status: OPTIMAL
Objective value: 75730195.0
Best bound: 75730195.0
Status: OPTIMAL
Objective value: 45370236.0
Best bound: 45370236.0
Status: OPTIMAL
Objective value: 81771369.0
Best bound: 81771369.0
Status: OPTIMAL
Objective value: 80775472.0
Best bound: 80775472.0
Status: OPTIMAL
Objective value: 91874445.0
Best bound: 91874445.0


In [25]:
display(result)

MultiScheduleResult(detailed_schedule=    schedule_date     sales_id territory_id   customer_id  \
0      2026-07-01  SAL_DMM_001      TER_DMM  CUS_DMM_0012   
1      2026-07-01  SAL_DMM_001      TER_DMM  CUS_DMM_0032   
2      2026-07-01  SAL_DMM_001      TER_DMM  CUS_DMM_0047   
3      2026-07-01  SAL_DMM_001      TER_DMM  CUS_DMM_0080   
4      2026-07-01  SAL_DMM_001      TER_DMM  CUS_DMM_0096   
..            ...          ...          ...           ...   
655    2026-07-27  SAL_RUH_003      TER_RUH  CUS_RUH_0080   
656    2026-07-28  SAL_RUH_003      TER_RUH  CUS_RUH_0002   
657    2026-07-28  SAL_RUH_003      TER_RUH  CUS_RUH_0005   
658    2026-07-30  SAL_RUH_003      TER_RUH  CUS_RUH_0051   
659    2026-07-30  SAL_RUH_003      TER_RUH  CUS_RUH_0088   

                               shop_name      locality    gps_lat    gps_lng  \
0                Al Riyadh Corner Market    Al Badiyah  26.397293  50.045975   
1                    Al Saleh Fresh Meat  Al Mazruiyah  26.455036  50

In [27]:
import pandas as pd

# Define the output file name
excel_file_path = "monthly_plan.xlsx"

with pd.ExcelWriter(excel_file_path, engine="openpyxl") as writer:
    # 1. Save the main aggregated dataframes to their own sheets
    result.detailed_schedule.to_excel(
        writer, sheet_name="Detailed Schedule", index=False
    )
    result.daily_schedule.to_excel(writer, sheet_name="Daily Schedule", index=False)

    # 2. Loop through the salesperson dictionary and write each to a sheet
    for name, sp_result in result.salesperson_results.items():
        # Excel sheet names cannot exceed 31 characters
        safe_sheet_name = f"SP_{name}"

        # Check if the salesperson result object itself has dataframes to unpack
        # If it is a dictionary, use: sp_result_dict = sp_result
        # If it is another dataclass, convert it to a dict to read its attributes
        if hasattr(sp_result, "__dataclass_fields__"):
            from dataclasses import asdict

            sp_data = asdict(sp_result)
        elif isinstance(sp_result, dict):
            sp_data = sp_result
        else:
            # Fallback: tries to extract internal attributes if it's a custom class
            sp_data = vars(sp_result)

        # Look for any pandas DataFrames inside the salesperson's data to save
        for key, value in sp_data.items():
            if isinstance(value, pd.DataFrame):
                # Creates a unique sheet name like "SP_John_detailed"
                sub_sheet_name = f"SP_{name}_{key}"
                value.to_excel(writer, sheet_name=sub_sheet_name, index=False)

print(f"Successfully saved results to {excel_file_path}")


d:\Data Science\Basamh\JP\journey-planner\.venv\Lib\site-packages\openpyxl\workbook\child.py:99: UserWarning: Title is more than 31 characters. Some applications may not be able to read the file
  warnings.warn("Title is more than 31 characters. Some applications may not be able to read the file")


Successfully saved results to monthly_plan.xlsx


In [ ]:
from saudi_multi_salesperson_scheduler import MultiSalespersonScheduler, build_route_map_for_salesperson
scheduler = MultiSalespersonScheduler(
    extra_champion_bonus=30,
    solver_time_seconds=60,
)

result = scheduler.create_schedules(
    customer_df      = customer_df,
    rfm_scores_df    = rfm_scores_df,
    salesperson_df   = salesperson_df,
    holiday_df       = holiday_df,
    territory_df     = territory_df,
    config_df        = config_df,
    van_df           = van_df,
    month_start_date = "2026-06-01",
)

print(result.detailed_schedule.head(20))
print(result.daily_schedule.head(20))

for sales_id, sp in result.salesperson_results.items():
    print(f"{sales_id} → {len(sp.assigned_customers)} customers, {len(sp.detailed_schedule)} visits")
    
m = build_route_map_for_salesperson(
    daily_schedule    = result.daily_schedule,
    detailed_schedule = result.detailed_schedule,
    sales_id          = "SAL_DMM_001",
    schedule_date     = "2026-06-14",
)
m

Status: OPTIMAL
Objective value: 64656255.0
Best bound: 64656255.0
Status: OPTIMAL
Objective value: 43551210.0
Best bound: 43551210.0
Status: OPTIMAL
Objective value: 64630255.0
Best bound: 64630255.0
Status: OPTIMAL
Objective value: 48447255.0
Best bound: 48447255.0
Status: OPTIMAL
Objective value: 45370210.0
Best bound: 45370210.0
Status: OPTIMAL
Objective value: 80770315.0
Best bound: 80770315.0
Status: OPTIMAL
Objective value: 80775405.0
Best bound: 80775405.0
Status: OPTIMAL
Objective value: 75808240.0
Best bound: 75808240.0
   schedule_date     sales_id territory_id   customer_id  \
0     2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0001   
1     2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0002   
2     2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0019   
3     2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0080   
4     2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0098   
5     2026-06-14  SAL_DMM_001      TER_DMM  CUS_DMM_0012   
6     2026-06-14  SAL_DMM_001      TER_DMM  

In [ ]:
m = build_route_map_for_salesperson(
    daily_schedule    = result.daily_schedule,
    detailed_schedule = result.detailed_schedule,
    sales_id          = "SAL_JED_001",
    schedule_date     = "2026-06-30",
)
m

In [ ]:
m = build_route_map_for_salesperson(
    daily_schedule    = result.daily_schedule,
    detailed_schedule = result.detailed_schedule,
    sales_id          = "SAL_RUH_003",
    schedule_date     = "2026-06-09",
)
m

In [ ]:
result

MultiScheduleResult(detailed_schedule=    schedule_date     sales_id territory_id   customer_id  \
0      2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0001   
1      2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0019   
2      2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0064   
3      2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0080   
4      2026-06-09  SAL_DMM_001      TER_DMM  CUS_DMM_0098   
..            ...          ...          ...           ...   
494    2026-06-29  SAL_RUH_003      TER_RUH  CUS_RUH_0072   
495    2026-06-30  SAL_RUH_003      TER_RUH  CUS_RUH_0002   
496    2026-06-30  SAL_RUH_003      TER_RUH  CUS_RUH_0005   
497    2026-06-30  SAL_RUH_003      TER_RUH  CUS_RUH_0017   
498    2026-06-30  SAL_RUH_003      TER_RUH  CUS_RUH_0085   

                         shop_name         locality    gps_lat    gps_lng  \
0     Al Shehri Trading Fresh Meat     Al Mazruiyah  26.456625  50.084601   
1    Al Tazaj Hospitality Supplies       Al Badiyah  26.431079  50.059445  